# Stored execution evidence — TGS Salt Identification Challenge (template)

This public evidence copy preserves every textual output cell supplied in `mle_final_submission.zip`.
The original notebook SHA-256 is `7d5983c6720825d63e86b6f04ae93536f0c278144aa5680a0d5fee93b64e205c`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Standalone MLE-STAR benchmark runner — TGS Salt Identification

This notebook preserves the same experiment contract used by the other formal MLE-STAR notebooks:

1. reproducible environment and Kaggle data setup;
2. three paired seeds (`13, 29, 47`);
3. four reported arms (`baseline`, `mlestar_initial`, `mlestar_refined`, `mlestar_ensemble`);
4. identical folds within each seed;
5. official TGS mean precision over IoU thresholds `0.50:0.05:0.95`;
6. `comparison.csv`, checkpoints, epoch history, `submission.csv`, and optional Kaggle API submission.

TGS is a **Custom Extension**, not one of the 22 official MLE-bench Lite tasks. The notebook therefore adds only the segmentation adapter required for this competition; it does not change the MLE-STAR comparison protocol.

In [ ]:
# Reproducible environment setup.
import os
import sys
import subprocess

packages = [
    "kaggle>=2.2.0",
    "segmentation-models-pytorch>=0.5.0",
    "timm>=1.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running the TGS experiment.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Credentials, controls, and Kaggle data download.
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import drive
from google.colab import userdata

drive.mount("/content/drive")

COMPETITION = "tgs-salt-identification-challenge"
DATA_ROOT = Path("/content/tgs-salt-identification-challenge")
RUNS_ROOT = Path("/content/drive/MyDrive/mlestar-runs")
RUN_ROOT = RUNS_ROOT / "tgs_salt"

SEEDS = (13, 29, 47)
N_SPLITS = 3
MAX_EPOCHS = 20
MIN_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 4
IMAGE_SIZE = 128
BATCH_SIZE = 16
NUM_WORKERS = 2

# Keep False when you want to resume completed fold checkpoints.
# Set True only when intentionally starting a completely new formal run.
FRESH_START = False

# Leave False until submission.csv has been inspected.
SUBMIT = False
SUBMISSION_MESSAGE = "MLE-STAR TGS Salt segmentation submission"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if FRESH_START and RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

token = userdata.get("KAGGLE_API_TOKEN")
if token:
    token_path = Path.home() / ".kaggle" / "access_token"
    token_path.parent.mkdir(parents=True, exist_ok=True)
    token_path.write_text(token.strip(), encoding="utf-8")
    token_path.chmod(0o600)
elif not (Path.home() / ".kaggle" / "kaggle.json").is_file():
    raise RuntimeError("Add KAGGLE_API_TOKEN to Colab Secrets and enable notebook access.")
del token


In [ ]:
# Reusable Kaggle competition fetcher and TGS archive extraction.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
marker = DATA_ROOT / "train.csv"
if not marker.is_file():
    subprocess.run(
        ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", str(DATA_ROOT)],
        check=True,
    )
    outer_zip = DATA_ROOT / f"{COMPETITION}.zip"
    with zipfile.ZipFile(outer_zip) as archive:
        archive.extractall(DATA_ROOT)

for archive_name, output_name in (("train.zip", "train"), ("test.zip", "test")):
    archive_path = DATA_ROOT / archive_name
    output_path = DATA_ROOT / output_name
    if archive_path.is_file() and not output_path.is_dir():
        output_path.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(output_path)

print("Competition:", COMPETITION)
print("Data root:", DATA_ROOT)
print("Run root:", RUN_ROOT)
print("Seeds:", SEEDS)
print("Folds:", N_SPLITS)
print("Maximum epochs:", MAX_EPOCHS)

In [ ]:
#@title TGS data and exact-resize cache (shared helper) { display-mode: "form" }
# TGS segmentation adapter and official metric (shared helper).
import gc
import hashlib
import math
import time
from dataclasses import dataclass

from PIL import Image
from sklearn.model_selection import KFold
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def locate_image_dirs(root: Path):
    image_dirs = [p.parent for p in root.rglob("*.png") if p.parent.name == "images"]
    mask_dirs = [p.parent for p in root.rglob("*.png") if p.parent.name == "masks"]
    image_dirs = sorted(set(image_dirs))
    mask_dirs = sorted(set(mask_dirs))
    train_images = next((p for p in image_dirs if "train" in str(p).lower()), None)
    test_images = next((p for p in image_dirs if "test" in str(p).lower()), None)
    train_masks = next((p for p in mask_dirs if "train" in str(p).lower()), None)
    if not train_images or not train_masks or not test_images:
        raise FileNotFoundError(
            f"Could not locate TGS train/images, train/masks and test/images under {root}. "
            f"images={image_dirs}, masks={mask_dirs}"
        )
    return train_images, train_masks, test_images


TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, TEST_IMAGE_DIR = locate_image_dirs(DATA_ROOT)
train_paths = sorted(TRAIN_IMAGE_DIR.glob("*.png"))
mask_paths = [TRAIN_MASK_DIR / p.name for p in train_paths]
test_paths = sorted(TEST_IMAGE_DIR.glob("*.png"))
missing_masks = [str(p) for p in mask_paths if not p.is_file()]
if missing_masks:
    raise FileNotFoundError(f"Missing masks, examples: {missing_masks[:5]}")


In [ ]:
#@title MLE-selected TGS segmentation adapter (shared helper) { display-mode: "form" }
class TGSDataset(Dataset):
    def __init__(self, images, masks=None, image_size=128, training=False):
        self.images = list(images)
        self.masks = None if masks is None else list(masks)
        self.image_size = int(image_size)
        self.training = bool(training)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        with Image.open(self.images[index]) as handle:
            image = handle.convert("RGB").resize(
                (self.image_size, self.image_size), Image.Resampling.BILINEAR
            )
            image = np.asarray(image, dtype=np.float32) / 255.0
        if self.masks is None:
            return torch.from_numpy(image).permute(2, 0, 1)
        with Image.open(self.masks[index]) as handle:
            mask = handle.convert("L").resize(
                (self.image_size, self.image_size), Image.Resampling.NEAREST
            )
            mask = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)
        if self.training and torch.rand(()) < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
        return torch.from_numpy(image).permute(2, 0, 1), torch.from_numpy(mask)[None]


def build_model(model_name):
    encoder_name = {"resnet18_unet": "resnet18", "efficientnet_b0_unet": "efficientnet-b0"}[model_name]
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )


def dice_bce_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probs * targets).sum(dims)
    denominator = probs.sum(dims) + targets.sum(dims)
    dice = 1.0 - ((2.0 * intersection + 1.0) / (denominator + 1.0)).mean()
    return 0.5 * bce + 0.5 * dice


def tgs_score_one(y_true, y_pred):
    true_sum = int(y_true.sum())
    pred_sum = int(y_pred.sum())
    if true_sum == 0 and pred_sum == 0:
        return 1.0
    if true_sum == 0 or pred_sum == 0:
        return 0.0
    intersection = float(np.logical_and(y_true, y_pred).sum())
    union = float(np.logical_or(y_true, y_pred).sum())
    iou = intersection / max(union, 1.0)
    return float(np.mean([iou > t for t in np.arange(0.50, 1.00, 0.05)]))


def tgs_score_batch(targets, predictions):
    return float(np.mean([tgs_score_one(t, p) for t, p in zip(targets, predictions)]))


def tune_threshold(targets, probabilities):
    best_threshold, best_score = 0.5, -1.0
    for threshold in np.arange(0.10, 0.901, 0.05):
        score = tgs_score_batch(targets, probabilities >= threshold)
        if score > best_score:
            best_threshold, best_score = float(round(threshold, 2)), float(score)
    return best_threshold, best_score


def predict_loader(model, loader):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for batch in loader:
            images = batch[0] if isinstance(batch, (tuple, list)) else batch
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                probs = torch.sigmoid(model(images))
            outputs.append(probs[:, 0].float().cpu().numpy())
    return np.concatenate(outputs)


def train_fold(model_name, seed, fold, train_idx, valid_idx):
    fold_root = RUN_ROOT / f"seed_{seed}" / model_name / f"fold_{fold}"
    fold_root.mkdir(parents=True, exist_ok=True)
    checkpoint_path = fold_root / "best_model.pt"
    history_path = fold_root / "history.json"

    valid_dataset = TGSDataset(
        [train_paths[i] for i in valid_idx], [mask_paths[i] for i in valid_idx],
        IMAGE_SIZE, False,
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    if checkpoint_path.is_file() and history_path.is_file() and not FRESH_START:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        model = build_model(model_name)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.to(DEVICE)
        probabilities = predict_loader(model, valid_loader)
        del model
        gc.collect(); torch.cuda.empty_cache()
        print(f"RESUME seed={seed} model={model_name} fold={fold}")
        return probabilities, checkpoint["best_epoch"], checkpoint["best_threshold"]

    torch.manual_seed(seed + fold)
    np.random.seed(seed + fold)
    model = build_model(model_name).to(DEVICE)
    train_dataset = TGSDataset(
        [train_paths[i] for i in train_idx], [mask_paths[i] for i in train_idx],
        IMAGE_SIZE, True,
    )
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(seed + fold),
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

    targets = np.stack([
        np.asarray(Image.open(mask_paths[i]).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST
        )) > 127 for i in valid_idx
    ])
    best_score, best_epoch, best_threshold = -1.0, 0, 0.5
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.perf_counter()
        model.train()
        running_loss = 0.0
        steps = 0
        for images, masks in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = dice_bce_loss(model(images), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach())
            steps += 1
        probabilities = predict_loader(model, valid_loader)
        threshold, score = tune_threshold(targets, probabilities)
        improved = score > best_score
        if improved:
            best_score, best_epoch, best_threshold = score, epoch, threshold
            stale = 0
            torch.save({
                "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "model_name": model_name,
                "seed": seed,
                "fold": fold,
                "best_epoch": best_epoch,
                "best_metric": best_score,
                "best_threshold": best_threshold,
                "metric_name": "tgs_mean_precision_iou_0.50_0.95",
                "image_size": IMAGE_SIZE,
            }, checkpoint_path)
        else:
            stale += 1
        elapsed = time.perf_counter() - started
        row = {
            "epoch": epoch, "loss": running_loss / max(steps, 1),
            "val_tgs_metric": score, "threshold": threshold,
            "best_epoch": best_epoch, "best_metric": best_score,
            "lr": optimizer.param_groups[0]["lr"], "seconds": elapsed,
        }
        history.append(row)
        history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
        print(
            f"[train] seed={seed} model={model_name} fold={fold + 1}/{N_SPLITS} "
            f"epoch={epoch}/{MAX_EPOCHS} loss={row['loss']:.6f} "
            f"val_tgs_metric={score:.6f} threshold={threshold:.2f} "
            f"best_epoch={best_epoch} best={best_score:.6f} time={elapsed:.1f}s",
            flush=True,
        )
        scheduler.step()
        if epoch >= MIN_EPOCHS and stale >= EARLY_STOPPING_PATIENCE:
            break

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    probabilities = predict_loader(model, valid_loader)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return probabilities, checkpoint["best_epoch"], checkpoint["best_threshold"]


print("Train images:", len(train_paths))
print("Test images:", len(test_paths))
print("Train image directory:", TRAIN_IMAGE_DIR)
print("Train mask directory:", TRAIN_MASK_DIR)
print("Test image directory:", TEST_IMAGE_DIR)

In [ ]:
#@title Benchmark runner (shared helper) { display-mode: "form" }
def run_tgs_oof_comparison():
    # Paired MLE-STAR comparison runner.
    MODEL_NAMES = ("resnet18_unet", "efficientnet_b0_unet")
    comparison_rows = []
    seed_contracts = []
    
    all_targets = np.stack([
        np.asarray(Image.open(path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST
        )) > 127 for path in mask_paths
    ])
    
    for seed in SEEDS:
        splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        splits = list(splitter.split(np.arange(len(train_paths))))
        candidate_oof = {}
        candidate_epochs = {}
    
        for model_name in MODEL_NAMES:
            oof = np.zeros((len(train_paths), IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
            selected_epochs = []
            for fold, (train_idx, valid_idx) in enumerate(splits):
                fold_probs, best_epoch, _ = train_fold(
                    model_name, seed, fold, train_idx, valid_idx
                )
                oof[valid_idx] = fold_probs
                selected_epochs.append(int(best_epoch))
            candidate_oof[model_name] = oof
            candidate_epochs[model_name] = selected_epochs
    
        thresholds = {}
        scores = {}
        for model_name, oof in candidate_oof.items():
            thresholds[model_name], scores[model_name] = tune_threshold(all_targets, oof)
    
        baseline_model = MODEL_NAMES[0]
        initial_model = max(MODEL_NAMES, key=lambda name: scores[name])
        # Refinement tests the alternate model on the same folds and retains the better candidate.
        refined_model = max((initial_model, MODEL_NAMES[1]), key=lambda name: scores[name])
    
        best_weight, best_ensemble_score, best_ensemble_threshold = 0.0, -1.0, 0.5
        for refined_weight in np.arange(0.0, 1.01, 0.1):
            ensemble_oof = (
                (1.0 - refined_weight) * candidate_oof[baseline_model]
                + refined_weight * candidate_oof[refined_model]
            )
            threshold, score = tune_threshold(all_targets, ensemble_oof)
            if score > best_ensemble_score:
                best_weight = float(round(refined_weight, 1))
                best_ensemble_score = float(score)
                best_ensemble_threshold = float(threshold)
    
        arm_values = {
            "baseline": scores[baseline_model],
            "mlestar_initial": scores[initial_model],
            "mlestar_refined": scores[refined_model],
            "mlestar_ensemble": best_ensemble_score,
        }
        comparison_rows.extend(
            {"seed": seed, "arm": arm, "metric_value": value}
            for arm, value in arm_values.items()
        )
        contract = {
            "seed": seed,
            "baseline_model": baseline_model,
            "initial_model": initial_model,
            "refined_model": refined_model,
            "ensemble_refined_weight": best_weight,
            "ensemble_threshold": best_ensemble_threshold,
            "selected_epochs": candidate_epochs,
            "metric": "tgs_mean_precision_iou_0.50_0.95",
            "paired_folds": True,
        }
        seed_contracts.append(contract)
        contract_path = RUN_ROOT / f"seed_{seed}" / "mle_star_contract.json"
        contract_path.write_text(json.dumps(contract, indent=2), encoding="utf-8")
    
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(RUN_ROOT / "comparison.csv", index=False)
    
    summary = comparison.groupby("arm")["metric_value"].agg(["mean", "sem", "count"])
    report = {
        "benchmark": "tgs_salt",
        "competition": COMPETITION,
        "metric": "tgs_mean_precision_iou_0.50_0.95",
        "paired_folds": True,
        "seeds": list(SEEDS),
        "arms": ["baseline", "mlestar_initial", "mlestar_refined", "mlestar_ensemble"],
        "status": "offline_oof_complete",
        "summary": summary.reset_index().to_dict(orient="records"),
    }
    (RUN_ROOT / "comparison.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    
    print(json.dumps(report, indent=2))
    display(comparison)
    display(summary)
    return report, comparison, seed_contracts


In [ ]:
#@title Kaggle prediction and submission (shared helper) { display-mode: "form" }
def run_tgs_kaggle_prediction(seed_contracts):
    # Final MLE-selected fit, TGS test prediction, submission validation and optional API upload.
    def fit_full_and_predict(model_name, seed, epochs):
        final_root = RUN_ROOT / f"final_seed_{seed}" / model_name
        final_root.mkdir(parents=True, exist_ok=True)
        checkpoint_path = final_root / "final_model.pt"
        prediction_path = final_root / "test_probabilities.npy"
        if prediction_path.is_file() and checkpoint_path.is_file() and not FRESH_START:
            print(f"RESUME final test probabilities: seed={seed} model={model_name}")
            return np.load(prediction_path)
    
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = build_model(model_name).to(DEVICE)
        dataset = TGSDataset(train_paths, mask_paths, IMAGE_SIZE, True)
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True,
            generator=torch.Generator().manual_seed(seed),
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
        for epoch in range(1, int(epochs) + 1):
            started = time.perf_counter()
            model.train()
            running_loss, steps = 0.0, 0
            for images, masks in loader:
                images = images.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    loss = dice_bce_loss(model(images), masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.detach()); steps += 1
            print(
                f"[final] seed={seed} model={model_name} epoch={epoch}/{epochs} "
                f"loss={running_loss / max(steps, 1):.6f} "
                f"time={time.perf_counter() - started:.1f}s",
                flush=True,
            )
        torch.save({
            "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "model_name": model_name, "seed": seed, "epochs": int(epochs),
            "image_size": IMAGE_SIZE,
        }, checkpoint_path)
        test_loader = DataLoader(
            TGSDataset(test_paths, None, IMAGE_SIZE, False),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
        )
        probabilities = predict_loader(model, test_loader)
        np.save(prediction_path, probabilities)
        del model
        gc.collect(); torch.cuda.empty_cache()
        return probabilities
    
    
    seed_predictions = []
    seed_thresholds = []
    for contract in seed_contracts:
        seed = int(contract["seed"])
        baseline_model = contract["baseline_model"]
        refined_model = contract["refined_model"]
        weight = float(contract["ensemble_refined_weight"])
    
        def selected_epoch(model_name):
            values = contract["selected_epochs"][model_name]
            return max(1, int(round(float(np.median(values)))))
    
        components = []
        weights = []
        if 1.0 - weight > 0:
            components.append(fit_full_and_predict(
                baseline_model, seed, selected_epoch(baseline_model)
            ))
            weights.append(1.0 - weight)
        if weight > 0:
            components.append(fit_full_and_predict(
                refined_model, seed, selected_epoch(refined_model)
            ))
            weights.append(weight)
        seed_predictions.append(sum(w * p for w, p in zip(weights, components)))
        seed_thresholds.append(float(contract["ensemble_threshold"]))
    
    test_probabilities = np.mean(seed_predictions, axis=0)
    final_threshold = float(np.median(seed_thresholds))
    
    
    def rle_encode(mask):
        pixels = np.asarray(mask, dtype=np.uint8).T.flatten()
        pixels = np.concatenate([[0], pixels, [0]])
        runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
        runs[1::2] -= runs[::2]
        return " ".join(str(int(value)) for value in runs)
    
    
    rles = []
    for probability in test_probabilities:
        resized = np.asarray(
            Image.fromarray(probability.astype(np.float32), mode="F").resize(
                (101, 101), Image.Resampling.BILINEAR
            ), dtype=np.float32,
        )
        rles.append(rle_encode(resized >= final_threshold))
    
    sample_path = DATA_ROOT / "sample_submission.csv"
    sample = pd.read_csv(sample_path)
    id_column = sample.columns[0]
    prediction_column = sample.columns[1]
    predictions_by_id = {path.stem: rle for path, rle in zip(test_paths, rles)}
    missing = sorted(set(sample[id_column].astype(str)) - set(predictions_by_id))
    if missing:
        raise RuntimeError(f"Test predictions are missing IDs, examples: {missing[:5]}")
    
    submission = sample.copy()
    submission[prediction_column] = submission[id_column].astype(str).map(predictions_by_id)
    if submission[prediction_column].isna().any():
        raise RuntimeError("submission.csv contains missing RLE values")
    
    submission_path = RUN_ROOT / "submission.csv"
    submission.to_csv(submission_path, index=False)
    local_copy = Path("/content/tgs_salt_submission.csv")
    submission.to_csv(local_copy, index=False)
    
    print("=== KAGGLE SUBMISSION READY ===")
    print("file:", submission_path)
    print("local copy:", local_copy)
    print("shape:", submission.shape)
    print("threshold:", final_threshold)
    print("empty masks:", int((submission[prediction_column].str.len() == 0).sum()))
    display(submission.head())
    
    if SUBMIT:
        subprocess.run([
            "kaggle", "competitions", "submit",
            "-c", COMPETITION,
            "-f", str(submission_path),
            "-m", SUBMISSION_MESSAGE,
        ], check=True)
        subprocess.run(["kaggle", "competitions", "submissions", "-c", COMPETITION], check=True)
    else:
        print("SUBMIT=False: CSV was validated but not uploaded.")
    return submission


In [ ]:
# TGS Salt Identification Challenge.
DATA_ROOT = Path("/content/tgs-salt-identification-challenge")
RUN_ROOT = RUNS_ROOT / "tgs_salt"

# Paired OOF comparison -> MLE ensemble reconstruction -> test prediction ->
# sample_submission validation -> optional Kaggle API submission.
report, result, seed_contracts = run_tgs_oof_comparison()
submission = run_tgs_kaggle_prediction(seed_contracts)
pd.read_csv(RUN_ROOT / "comparison.csv") if (RUN_ROOT / "comparison.csv").is_file() else result

In [ ]:
# Compact MLE-STAR experiment and Kaggle-submission table.
summary = pd.DataFrame([
    {
        "benchmark": "tgs_salt",
        "task_origin": "Custom Extension",
        "metric": "TGS mean precision over IoU thresholds 0.50:0.05:0.95",
        "paired_folds": True,
        "seeds": list(SEEDS),
        "arms": ["baseline", "mlestar_initial", "mlestar_refined", "mlestar_ensemble"],
        "comparison_csv": str(RUN_ROOT / "comparison.csv"),
        "submission_csv": str(RUN_ROOT / "submission.csv"),
        "submitted_by_api": bool(SUBMIT),
    }
])
display(summary)

## Output contract

The formal experiment artifacts are stored under:

`/content/drive/MyDrive/mlestar-runs/tgs_salt`

Important files:

- `comparison.csv`: three-seed, four-arm paired comparison;
- `comparison.json`: experiment summary and protocol metadata;
- `seed_*/mle_star_contract.json`: selected models, ensemble weights and epochs;
- `seed_*/.../fold_*/history.json`: per-epoch histories;
- `seed_*/.../fold_*/best_model.pt`: fold checkpoints;
- `submission.csv`: validated Kaggle submission.

The Kaggle score is separate from the offline comparison. Set `SUBMIT=True` only after reviewing the generated CSV.